In [ ]:
# Experiment 4 - Car Price Prediction MLOps
# Student: Bhavani Prasad
# GitHub: bhavaniprasad
# Email: bmudili918@gmail.com

from pathlib import Path
import os
import shutil

# Keep the notebook and car_prediction_data.csv in the same working folder.
project_path = Path.cwd()
dataset_name = "car_prediction_data.csv"

print("Project folder:", project_path)
print("Dataset path:", project_path / dataset_name)


In [ ]:
%pwd

In [ ]:
!cd

In [ ]:
# Verify that the dataset is available before starting.
from pathlib import Path

dataset_path = Path("car_prediction_data.csv")

if not dataset_path.exists():
    raise FileNotFoundError(
        "car_prediction_data.csv was not found in the current Jupyter folder. "
        "Place the CSV beside this notebook and run again."
    )

print("Dataset found:", dataset_path.resolve())


In [ ]:
!dir

In [ ]:
# Initialize Git
!git init
!git status


In [ ]:
# Configure Git identity
!git config user.name "bhavaniprasad"
!git config user.email "bmudili918@gmail.com"
!git branch -M main
!git status


In [ ]:
# Connect this local repository to your GitHub repository
!git remote add origin https://github.com/bhavaniprasadmudili/carprice-prediction.git
!git remote -v


In [ ]:
# Initialize DVC
!dvc init
!dvc --version
!dir


In [ ]:
# Track the dataset with DVC
!dvc add "car_prediction_data.csv"
!dir


In [ ]:
# Commit DVC metadata to Git
!git add "car_prediction_data.csv.dvc"
!git add .dvc/config .dvcignore
!git status
!git commit -m "Initialize Experiment 4 with DVC dataset tracking"


In [ ]:
# Push initial repository state to GitHub
!git push -u origin main


## STAGE 1 — DATA INGESTION

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
data = pd.read_csv("car_prediction_data.csv")


In [ ]:
print("Dataset Shape:", data.shape)


In [ ]:
data.head()


In [ ]:
print(data.columns.tolist())


In [ ]:
data.info()


## STAGE 2 — DATA PROFILING

In [ ]:
data.info()


In [ ]:
data.describe()


In [ ]:
data.describe(include="all")


In [ ]:
print(data.dtypes)


In [ ]:
print(data.isnull().sum())


In [ ]:
missing_percentage = (data.isnull().sum() / len(data)) * 100
print(missing_percentage)


In [ ]:
print("Duplicate Rows:", data.duplicated().sum())


In [ ]:
for column in data.columns:
    print(column, ":", data[column].nunique())


In [ ]:
print("Number of Rows:", data.shape[0])
print("Number of Columns:", data.shape[1])


## NEXT → STAGE 3: DATA VALIDATION

In [ ]:
print("Dataset Shape:", data.shape)


In [ ]:
missing_values = data.isnull().sum()
print("Missing Values:")
print(missing_values)


In [ ]:
duplicate_rows = data.duplicated().sum()
print("Duplicate Rows:", duplicate_rows)


In [ ]:
print(data.dtypes)


In [ ]:
numerical_columns = data.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = data.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical Columns:", numerical_columns)
print("Categorical Columns:", categorical_columns)


In [ ]:
for column in data.columns:
    print(column, "Unique Values:", data[column].nunique())


In [ ]:
print("Total Rows:", data.shape[0])
print("Total Columns:", data.shape[1])
print("Total Missing Values:", data.isnull().sum().sum())
print("Total Duplicate Rows:", data.duplicated().sum())


## STAGE 4 — DATA CLEANING

In [ ]:
cleaned_data = data.copy()


In [ ]:
cleaned_data = cleaned_data.drop_duplicates()

print("Dataset Shape:", cleaned_data.shape)


In [ ]:
numerical_columns = cleaned_data.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = cleaned_data.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical Columns:", numerical_columns)
print("Categorical Columns:", categorical_columns)


In [ ]:
for column in numerical_columns:
    cleaned_data[column] = cleaned_data[column].fillna(
        cleaned_data[column].median()
    )


In [ ]:
for column in categorical_columns:
    if cleaned_data[column].isnull().sum() > 0:
        cleaned_data[column] = cleaned_data[column].fillna(
            cleaned_data[column].mode()[0]
        )


In [ ]:
print("Missing Values:")
print(cleaned_data.isnull().sum())

print("Duplicate Rows:", cleaned_data.duplicated().sum())

print("Final Dataset Shape:", cleaned_data.shape)


## NEXT → STAGE 5: DATA TRANSFORMATION

In [ ]:
transformed_data = cleaned_data.copy()


In [ ]:
categorical_columns = transformed_data.select_dtypes(
    include=["object"]
).columns.tolist()

print(categorical_columns)


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}

for column in categorical_columns:
    encoder = LabelEncoder()
    transformed_data[column] = encoder.fit_transform(
        transformed_data[column].astype(str)
    )
    label_encoders[column] = encoder


In [ ]:
transformed_data.head()


In [ ]:
print(transformed_data.dtypes)


In [ ]:
print(transformed_data.isnull().sum())


In [ ]:
print("Dataset Shape:", transformed_data.shape)
print("Columns:", transformed_data.columns.tolist())


## STAGE 6 — FEATURE ENGINEERING

In [ ]:
feature_data = transformed_data.copy()


In [ ]:
# Create car age from the manufacturing year.
from datetime import datetime

current_year = datetime.now().year

feature_data["Car_Age"] = (
    current_year - feature_data["Year"]
)


In [ ]:
# Create a price-per-kilometre feature.
feature_data["Price_per_Km"] = (
    feature_data["Present_Price"] /
    feature_data["Kms_Driven"].replace(0, np.nan)
)

feature_data["Price_per_Km"] = (
    feature_data["Price_per_Km"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(feature_data["Price_per_Km"].median())
)


In [ ]:
feature_data.head()


In [ ]:
print(feature_data.columns.tolist())


In [ ]:
print("Feature Data Shape:", feature_data.shape)


## STAGE 7 — PREPARE FEATURES (X) AND TARGET (y)

In [ ]:
X = feature_data.drop("Selling_Price", axis=1)

y = feature_data["Selling_Price"]


In [ ]:
print("Features Shape:", X.shape)
print("Target Shape:", y.shape)


In [ ]:
X.head()


In [ ]:
y.head()


In [ ]:
print(X.columns.tolist())


## STAGE 8: TRAIN-TEST SPLIT

In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


In [ ]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


In [ ]:
X_train.head()


In [ ]:
X_test.head()


## STAGE 9 — DATA PREPROCESSING / FEATURE SCALING

In [ ]:
from sklearn.preprocessing import StandardScaler


In [ ]:
scaler = StandardScaler()


In [ ]:
X_train_scaled = scaler.fit_transform(X_train)


In [ ]:
X_test_scaled = scaler.transform(X_test)


In [ ]:
print("Scaled Training Shape:", X_train_scaled.shape)
print("Scaled Testing Shape:", X_test_scaled.shape)


In [ ]:
print(X_train_scaled[:5])


## STAGE 10: MODEL TRAINING

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor


In [ ]:
linear_model = LinearRegression()

random_forest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

gradient_boosting_model = GradientBoostingRegressor(
    random_state=42
)


In [ ]:
linear_model.fit(X_train_scaled, y_train)


In [ ]:
random_forest_model.fit(X_train_scaled, y_train)


In [ ]:
gradient_boosting_model.fit(X_train_scaled, y_train)


In [ ]:
linear_predictions = linear_model.predict(X_test_scaled)

random_forest_predictions = random_forest_model.predict(X_test_scaled)

gradient_boosting_predictions = gradient_boosting_model.predict(
    X_test_scaled
)


## STAGE 11: MODEL EVALUATION AND COMPARISON

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd


In [ ]:
linear_mae = mean_absolute_error(y_test, linear_predictions)
linear_rmse = mean_squared_error(y_test, linear_predictions) ** 0.5
linear_r2 = r2_score(y_test, linear_predictions)

print("Linear Regression")
print("MAE:", linear_mae)
print("RMSE:", linear_rmse)
print("R2 Score:", linear_r2)


In [ ]:
rf_mae = mean_absolute_error(y_test, random_forest_predictions)
rf_rmse = mean_squared_error(y_test, random_forest_predictions) ** 0.5
rf_r2 = r2_score(y_test, random_forest_predictions)

print("Random Forest")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R2 Score:", rf_r2)


In [ ]:
gb_mae = mean_absolute_error(y_test, gradient_boosting_predictions)
gb_rmse = mean_squared_error(y_test, gradient_boosting_predictions) ** 0.5
gb_r2 = r2_score(y_test, gradient_boosting_predictions)

print("Gradient Boosting")
print("MAE:", gb_mae)
print("RMSE:", gb_rmse)
print("R2 Score:", gb_r2)


In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE": [
        linear_mae,
        rf_mae,
        gb_mae
    ],
    "RMSE": [
        linear_rmse,
        rf_rmse,
        gb_rmse
    ],
    "R2 Score": [
        linear_r2,
        rf_r2,
        gb_r2
    ]
})

results


In [ ]:
results.sort_values(by="R2 Score", ascending=False)


## STAGE 12: BEST MODEL SELECTION

In [ ]:
best_model_name = results.loc[
    results["R2 Score"].idxmax(),
    "Model"
]

print("Best Model:", best_model_name)


In [ ]:
best_model_results = results.loc[
    results["R2 Score"].idxmax()
]

print(best_model_results)


In [ ]:
if best_model_name == "Linear Regression":
    best_model = linear_model
    best_predictions = linear_predictions
elif best_model_name == "Random Forest":
    best_model = random_forest_model
    best_predictions = random_forest_predictions
else:
    best_model = gradient_boosting_model
    best_predictions = gradient_boosting_predictions


In [ ]:
best_mae = best_model_results["MAE"]
best_rmse = best_model_results["RMSE"]
best_r2 = best_model_results["R2 Score"]

print("Best Model:", best_model_name)
print("Best MAE:", best_mae)
print("Best RMSE:", best_rmse)
print("Best R2 Score:", best_r2)


## STAGE 13 — MLFLOW EXPERIMENT TRACKING

In [ ]:
!pip install mlflow


In [ ]:
import mlflow
import mlflow.sklearn

# Use a SQLite backend so local Model Registry works reliably.
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Car_Price_Prediction_Experiment")


In [ ]:
with mlflow.start_run(run_name=f"{best_model_name}_car_price"):

    mlflow.log_param("model_name", best_model_name)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)

    if best_model_name == "Random Forest":
        mlflow.log_param("n_estimators", 100)

    mlflow.log_metric("MAE", float(best_mae))
    mlflow.log_metric("RMSE", float(best_rmse))
    mlflow.log_metric("R2_Score", float(best_r2))

print("Experiment tracked successfully!")


In [ ]:
# Log the selected best model in a separate MLflow run.
with mlflow.start_run(run_name=f"{best_model_name}_model") as run:

    mlflow.log_param("model_name", best_model_name)
    mlflow.log_metric("MAE", float(best_mae))
    mlflow.log_metric("RMSE", float(best_rmse))
    mlflow.log_metric("R2_Score", float(best_r2))

    mlflow.sklearn.log_model(
        sk_model=best_model,
        artifact_path="car_price_model"
    )

    model_run_id = run.info.run_id

print("Model and metrics tracked successfully!")
print("Model Run ID:", model_run_id)


In [ ]:
runs = mlflow.search_runs(
    experiment_names=["Car_Price_Prediction_Experiment"]
)

runs[["run_id", "status", "artifact_uri"]]


In [ ]:
# Open MLflow UI from a separate Anaconda Prompt / terminal:
# mlflow ui --port 5000 --backend-store-uri sqlite:///mlflow.db

print("Run the command above in a separate terminal, then open http://127.0.0.1:5000")


## STAGE 14 — MLFLOW MODEL REGISTRY

In [ ]:
model_uri = f"runs:/{model_run_id}/car_price_model"

print("Model URI:", model_uri)


In [ ]:
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name="CarPricePredictionModel"
)

print("Model Name:", registered_model.name)
print("Model Version:", registered_model.version)


## STAGE 15 — MODEL COMPARISON

In [ ]:
print(results)

comparison_results = results.sort_values(
    by="R2 Score",
    ascending=False
)

comparison_results


In [ ]:
print("Best Model:", best_model_name)
print("Best MAE:", best_mae)
print("Best RMSE:", best_rmse)
print("Best R2 Score:", best_r2)


In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    results["Model"],
    results["R2 Score"]
)

plt.xlabel("Models")
plt.ylabel("R2 Score")
plt.title("Car Price Model Comparison using R2 Score")

plt.xticks(rotation=15)

plt.show()


## STAGE 16 — DEPLOYMENT / CAR PRICE PREDICTION

In [ ]:
print(X.columns.tolist())


In [ ]:
# Example input based on a real category/value present in the supplied dataset.
new_car = pd.DataFrame({
    "Car_Name": ["ritz"],
    "Year": [2014],
    "Present_Price": [5.59],
    "Kms_Driven": [27000],
    "Fuel_Type": ["Petrol"],
    "Seller_Type": ["Dealer"],
    "Transmission": ["Manual"],
    "Owner": [0]
})

new_car


In [ ]:
# Apply the same label encoders used during training.
for column in categorical_columns:
    new_car[column] = label_encoders[column].transform(
        new_car[column].astype(str)
    )


In [ ]:
# Apply the same feature engineering used for the training data.
new_car["Car_Age"] = (
    current_year - new_car["Year"]
)

new_car["Price_per_Km"] = (
    new_car["Present_Price"] /
    new_car["Kms_Driven"].replace(0, np.nan)
)

new_car["Price_per_Km"] = (
    new_car["Price_per_Km"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(feature_data["Price_per_Km"].median())
)


In [ ]:
new_car = new_car[X.columns]

print(new_car.columns.tolist())


In [ ]:
new_car_scaled = scaler.transform(new_car)


In [ ]:
predicted_price = best_model.predict(new_car_scaled)

print("Predicted Car Selling Price:", predicted_price[0])


## STAGE 17 — FINAL GIT COMMIT AND PUSH

In [ ]:
!git status


In [ ]:
!dir *.ipynb


In [ ]:
!dir


In [ ]:
!git add .


In [ ]:
!git status


In [ ]:
!git commit -m "Complete Car Price Prediction MLOps pipeline"


In [ ]:
!git push origin main


## RESULT / OBSERVATION

The Car Price Prediction MLOps pipeline was completed using Git, DVC, MLflow and MLflow Model Registry.

Record the final values produced by your execution:
- Best Model: ______________________
- Best MAE: ______________________
- Best RMSE: ______________________
- Best R2 Score: ______________________
- Predicted Selling Price: ______________________
- MLflow Model Version: ______________________

**Important:** Do not manually enter MLflow run IDs, model versions, metrics or prediction values. Copy them from the outputs generated by the executed notebook.
